In [2]:
import pandas as pd
import numpy  as np
import torch

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"GPU is enabled. Using {torch.cuda.get_device_name(0)}.")
else:
    print("GPU is not enabled. Using CPU.")

GPU is enabled. Using Tesla T4.


In [4]:
from transformers import BertTokenizer, BertModel, BertConfig
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-large-portuguese-cased')
model = BertForSequenceClassification.from_pretrained('neuralmind/bert-large-portuguese-cased',
        num_labels=2 # Binary classification
)
model = model.to(device)

2025-04-29 15:38:22.920086: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745941103.151417      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745941103.221576      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


tokenizer_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-large-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
# Load the training data
train_df = pd.read_csv('/kaggle/input/tupy-e/binary_train.csv')
train_df = train_df[['text', 'hate','aggressive']].dropna()

# Load the test data
test_df = pd.read_csv('/kaggle/input/tupy-e/binary_test.csv')
test_df = test_df[['text', 'hate','aggressive']].dropna()


In [6]:
import re
def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()  # Lowercase
    text = re.sub(r'@\w+', '', text)  # Remove @mentions
    text = re.sub(r'http\S+|www.\S+', '', text)  # Remove URLs
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    return text.strip()
train_df['text'] = train_df['text'].apply(preprocess_text)
test_df['text'] = test_df['text'].apply(preprocess_text)

In [7]:
label = np.zeros((len(train_df['text']), 2))
for i in range(len(train_df)):
    if train_df['hate'][i] == 0:
        label[i] = [1, 0]
    else:
        label[i] = [0, 1]


In [8]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),
    label.tolist(),
    test_size=0.2,
    random_state=42
)


In [9]:
# For training and validation
from datasets import load_dataset, Dataset
train_dataset = Dataset.from_dict({'text': train_texts, 'labels': train_labels}) # Changed 'hate' to 'labels'
val_dataset = Dataset.from_dict({'text': val_texts, 'labels': val_labels}) # Changed 'hate' to 'labels'

# For testing
test_labels = [[1, 0] if label == 0 else [0, 1] for label in test_df['hate'].tolist()]
test_dataset = Dataset.from_dict({'text': test_df['text'].tolist(), 'labels': test_labels}) # Changed 'hate' to 'labels'

# Tokenize all
def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/27947 [00:00<?, ? examples/s]

Map:   0%|          | 0/6987 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

In [10]:


training_args = TrainingArguments(
    output_dir='./results',  # Directory to save the model checkpoints
    eval_strategy="epoch",  # Evaluate after each epoch
    
    logging_strategy="no",  # Disables logging
    save_strategy="epoch",  # Save model checkpoint after each epoch
    save_total_limit=2,  # Keep only the last 2 checkpoints
    num_train_epochs=10,  # Number of epochs to train
    per_device_train_batch_size=16,  # Batch size per device for training
    per_device_eval_batch_size=16,  # Batch size per device for evaluation
    learning_rate=2e-5,  # Learning rate for optimization
    weight_decay=0.01,  # Weight decay (for regularization)
    logging_dir='./logs',  # Directory to save logs
    load_best_model_at_end=True,  # Load the best model at the end of training
    metric_for_best_model='accuracy',  # Metric to track the best model
    report_to=[],  # Don't report metrics to any logging service
)


In [11]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    # Check if labels are multilabel-indicator (2D array)
    if labels.ndim == 2 and labels.shape[1] > 1:
        # If multilabel-indicator, convert to binary format
        labels = labels.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)


In [13]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.236183,0.901531,0.531335,0.612245,0.469314
2,No log,0.240166,0.903535,0.585995,0.598494,0.574007
3,No log,0.351518,0.880063,0.555202,0.496676,0.629362
4,No log,0.487329,0.895234,0.563766,0.558442,0.569194
5,No log,0.491308,0.894662,0.562426,0.555817,0.569194
6,No log,0.571217,0.901961,0.556060,0.602528,0.516245
7,No log,0.648390,0.901818,0.557990,0.600555,0.521059
8,No log,0.698991,0.898096,0.566910,0.573186,0.560770
9,No log,0.738669,0.899241,0.561097,0.582147,0.541516
10,No log,0.750231,0.901818,0.554545,0.602257,0.513839


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

TrainOutput(global_step=8740, training_loss=0.07017446557226116, metrics={'train_runtime': 14740.3131, 'train_samples_per_second': 18.96, 'train_steps_per_second': 0.593, 'total_flos': 6.511171452731904e+16, 'train_loss': 0.07017446557226116, 'epoch': 10.0})

In [14]:
trainer.save_model("./my-bertimbau-hate-large-model")
tokenizer.save_pretrained("./my-bertimbau-hate-large-model")


('./my-bertimbau-hate-large-model/tokenizer_config.json',
 './my-bertimbau-hate-large-model/special_tokens_map.json',
 './my-bertimbau-hate-large-model/vocab.txt',
 './my-bertimbau-hate-large-model/added_tokens.json')